# Решения: train/test и LR

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def find_trips_csv() -> Path:
    for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('trips.csv не найден')


TRIPS_PATH = find_trips_csv()
df = pd.read_csv(TRIPS_PATH)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


## Урок. 1. Leakage

In [ ]:
LEAKAGE_NOTE = (
    'Если учить и проверять на одних строках, ошибка занижена: модель видела ответы. '
    'Test — отложенные поездки, которых не было в fit.'
)
print(LEAKAGE_NOTE)

## Урок. 2. X и y

In [ ]:
X = df[['distance_km']]
y = df['duration_min']
print(X.shape, y.shape)

## Урок. 3. train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(len(X_train), len(X_test))

## Урок. 4. fit / predict

In [ ]:
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print('coef', model.coef_, 'intercept', model.intercept_)
print('первые 3 pred:', y_pred[:3])

## Урок. 5. Другой seed

In [ ]:
_, _, _, y_test_other = train_test_split(X, y, test_size=0.2, random_state=0)
same_first_indices = bool((y_test.index[:3] == y_test_other.index[:3]).all())
assert same_first_indices is False
print(same_first_indices)

## Урок. 6. Pipeline целиком

In [ ]:
X2 = df[['distance_km']]
y2 = df['duration_min']
Xtr, Xte, ytr, yte = train_test_split(X2, y2, test_size=0.2, random_state=42)
model2 = LinearRegression().fit(Xtr, ytr)
pred2 = model2.predict(Xte)
print('pipeline ok', len(Xtr), len(Xte), model2.coef_)

## Урок. 7. Checkpoint

In [ ]:
NO_FIT_ALL_NOTE = (
    'Ошибка на тех же строках, где был fit, не показывает обобщение на новые поездки.'
)
print(NO_FIT_ALL_NOTE)

## ДЗ. 1. test_size=0.3

In [ ]:
X = df[['distance_km']]
y = df['duration_min']
_, X_te, _, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
n_test_03 = len(X_te)
print(n_test_03)

## ДЗ. 2. Воспроизводимость

In [ ]:
a = train_test_split(X, y, test_size=0.2, random_state=42)[3].values
b = train_test_split(X, y, test_size=0.2, random_state=42)[3].values
reproducible = bool((a == b).all())
print(reproducible)

## ДЗ. 3. coef hour

In [ ]:
Xh = df[['hour']]
Xtr, Xte, ytr, yte = train_test_split(Xh, y, test_size=0.2, random_state=42)
coef_hour = LinearRegression().fit(Xtr, ytr).coef_
COEF_NOTE = 'Знак coef зависит от данных; на паре важнее зафиксировать seed и размер test'
print(coef_hour, COEF_NOTE)

## ДЗ. 4. Leakage

In [ ]:
HW_LEAKAGE = (
    'MSE на train после fit на тех же строках занижает ошибку. '
    'В отчёте нужны метрики на отложенном test.'
)
print(HW_LEAKAGE)